In [1]:
# ==============================
# 1. Imports
# ==============================

import os
import numpy as np
import faiss

from openai import AzureOpenAI
from dotenv import load_dotenv

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# ==============================
# 2. Load environment variables
# ==============================

load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")


In [ ]:
# ==============================
# 3. Load knowledge base
# ==============================

with open("../data/knowledge.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Each paragraph becomes one chunk
chunks = [p.strip() for p in text.split("\n\n") if p.strip()]

print("Chunks:")
for c in chunks:
    print("-", c)


In [ ]:
# ==============================
# 4. Create embeddings
# ==============================

def get_embedding(text):
    response = client.embeddings.create(
        model=embedding_deployment,
        input=text
    )
    return response.data[0].embedding


chunk_embeddings = [get_embedding(chunk) for chunk in chunks]

embedding_dim = len(chunk_embeddings[0])


# ==============================
# 5. Create FAISS index
# ==============================

index = faiss.IndexFlatL2(embedding_dim)

index.add(np.array(chunk_embeddings).astype("float32"))

print("FAISS index size:", index.ntotal)


In [ ]:
# ==============================
# 6. User question
# ==============================

question = "Which ocean is the largest on Earth?"

print("Question:", question)


# ==============================
# 7. Embed the question
# ==============================

question_embedding = np.array(
    [get_embedding(question)]
).astype("float32")



In [ ]:
# ==============================
# 8. Retrieve top 2 chunks
# ==============================

k = 2

distances, indices = index.search(question_embedding, k)

retrieved_chunks = [chunks[i] for i in indices[0]]

print("Retrieved context:")
for c in retrieved_chunks:
    print("-", c)


In [ ]:
# ==============================
# 9. Create RAG prompt
# ==============================

context = "\n\n".join(retrieved_chunks)

prompt = f"""
Answer the question using ONLY the provided context.

Context:
{context}

Question:
{question}

Answer:
"""


In [ ]:


# ==============================
# 10. Call GPT-4o
# ==============================

response = client.chat.completions.create(
    model=chat_deployment,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

print("\nLLM Answer:")
print(response.choices[0].message.content)